In [15]:
import os
import pandas as pd
import scanpy as sc
import loompy as lp
import json
import base64
import zlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import numpy as np
from pyscenic.rss import regulon_specificity_scores
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
sc.settings.njobs = 20

Name of the .loom file

In [3]:
f_loom_path_scenic = "TERVA.loom"

# Gene regulatory network inference, and generation of co-expression modules

## Load Transcription Factor Names

In [4]:
f_tfs = "./allTFs_mm.txt"

## Run Network Inference

> Output is a list of adjacencies connecting a TF with a target gene. A weight or importance is associated with these connections to distinguish strong from weak regulatory interactions (Van de Sande et al., 2020).

In [ ]:
!pyscenic grn {f_loom_path_scenic} {f_tfs} -o adj.csv --num_workers 20

# Module Generation

Define rankings and motif

In [6]:
f_db_glob = "./rankings/*feather"
f_db_names = ' '.join(glob.glob(f_db_glob) )

# motif databases
f_motif_path = "motif/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl"

In [ ]:
!pyscenic ctx adj.csv {f_db_names} --annotations_fname {f_motif_path} --expression_mtx_fname {f_loom_path_scenic} --output reg.csv --mask_dropouts --num_workers 20

# Cellular enrichment (AUCell)

In [8]:
# results will be saved in this file
f_pyscenic_output = "pyscenic_output.loom"

In [ ]:
!pyscenic aucell {f_loom_path_scenic} reg.csv --output {f_pyscenic_output} --num_workers 20

# Read the Output .loom File

In [10]:
loom = lp.connect(
    filename="pyscenic_output.loom",
    mode="r",
    validate=False
)

## Get AUC Matrix

In [12]:
loom.col_attrs.keys()

['CellID', 'Embedding', 'Embeddings_X', 'Embeddings_Y', 'RegulonsAUC', 'nGene']

In [13]:
auc_mtx = pd.DataFrame(
    loom.ca.RegulonsAUC,
    index=loom.ca.CellID
)

In [14]:
auc_mtx

,Ahr(+),Alx4(+),Ar(+),Arid3a(+),Arnt(+),Arnt2(+),Arx(+),Atf2(+),Atf3(+),Atf4(+),...,Zfp84(+),Zfp85(+),Zfp874a(+),Zfp93(+),Zfp942(+),Zfp947(+),Zfp952(+),Zfp955a(+),Zscan29(+),Zxdc(+)
OBAO_AACACGTAGTATGACA-1,0.000000,0.007912,0.168984,0.021288,0.050589,0.014560,0.221809,0.000000,0.052097,0.073967,...,0.0,0.073265,0.012657,0.00000,0.034517,0.026819,0.029678,0.000000,0.004337,0.0
OBAO_AACCATGAGCGCTTAT-1,0.000000,0.001047,0.156729,0.019026,0.034992,0.000000,0.160936,0.000000,0.097761,0.121861,...,0.0,0.075201,0.000000,0.00000,0.005476,0.005620,0.000000,0.000000,0.021113,0.0
OBAO_AACTCAGGTAGGCATG-1,0.000000,0.009629,0.117162,0.017905,0.048737,0.029616,0.000000,0.000000,0.104521,0.112025,...,0.0,0.024164,0.000000,0.00000,0.000000,0.011971,0.000000,0.000000,0.000000,0.0
OBAO_AACTGGTGTTACTGAC-1,0.000000,0.021317,0.124473,0.001817,0.041143,0.040768,0.000000,0.000000,0.095583,0.121765,...,0.0,0.035781,0.011285,0.00000,0.000000,0.000000,0.015242,0.000000,0.000000,0.0
OBAO_AACTTTCTCAACGAAA-1,0.000000,0.005543,0.186629,0.009670,0.039973,0.000000,0.000000,0.000000,0.115622,0.139658,...,0.0,0.000542,0.007022,0.00000,0.018987,0.000000,0.097831,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NOBPVAT_TTTGGTTCAAATTGCC-1,0.000000,0.007326,0.101103,0.017996,0.051373,0.000000,0.000000,0.009678,0.062009,0.076070,...,0.0,0.035626,0.007479,0.00000,0.009354,0.021154,0.022615,0.000000,0.010819,0.0
NOBPVAT_TTTGTCAAGATCGATA-1,0.000000,0.004764,0.110198,0.021449,0.043608,0.000000,0.000000,0.051981,0.035203,0.059714,...,0.0,0.067534,0.025595,0.30886,0.036475,0.028921,0.000000,0.000000,0.000143,0.0
NOBPVAT_TTTGTCAGTCATGCAT-1,0.000000,0.018671,0.193383,0.019806,0.035481,0.006444,0.000000,0.008097,0.038013,0.082031,...,0.0,0.021066,0.012229,0.00000,0.000000,0.000000,0.072862,0.000000,0.000000,0.0
NOBPVAT_TTTGTCATCGTTGCCT-1,0.063728,0.012643,0.110880,0.010213,0.033585,0.020384,0.080545,0.000000,0.072360,0.098141,...,0.0,0.000000,0.000251,0.00000,0.035336,0.012458,0.022243,0.000000,0.065056,0.0


* 350 regulons are predicted

* Save AUC matrix

In [32]:
auc_mtx.to_csv("auc_matrix.csv")

# Calculate Regulons Specificity Scores (RSS) Across Fibroblast Subclusters

## Read Meta Data

In [24]:
meta_data = pd.read_csv("./meta_data.csv")
meta_data.set_index("Unnamed: 0", drop=True, inplace=True)
meta_data.index.name = ""
meta_data.head()

,tissue_id,celltype,celltype.group,group_id
,,,,
OBAO_AACACGTAGTATGACA-1,Aorta,Cd248+ Pi16+ Fibroblasts,Cd248+ Pi16+ Fibroblasts_Obese,Obese
OBAO_AACCATGAGCGCTTAT-1,Aorta,Mfap4+ Fibroblasts,Mfap4+ Fibroblasts_Obese,Obese
OBAO_AACTCAGGTAGGCATG-1,Aorta,Cd248+ Pi16+ Fibroblasts,Cd248+ Pi16+ Fibroblasts_Obese,Obese
OBAO_AACTGGTGTTACTGAC-1,Aorta,Cd248+ Pi16+ Fibroblasts,Cd248+ Pi16+ Fibroblasts_Obese,Obese
OBAO_AACTTTCTCAACGAAA-1,Aorta,Mfap4+ Fibroblasts,Mfap4+ Fibroblasts_Obese,Obese


In [26]:
rss_subcluster = regulon_specificity_scores(auc_mtx=auc_mtx, cell_type_series=meta_data.celltype)

In [27]:
rss_subcluster

,Ahr(+),Alx4(+),Ar(+),Arid3a(+),Arnt(+),Arnt2(+),Arx(+),Atf2(+),Atf3(+),Atf4(+),...,Zfp84(+),Zfp85(+),Zfp874a(+),Zfp93(+),Zfp942(+),Zfp947(+),Zfp952(+),Zfp955a(+),Zscan29(+),Zxdc(+)
Cd248+ Pi16+ Fibroblasts,0.241028,0.359239,0.376346,0.373157,0.392758,0.353275,0.243563,0.308810,0.396011,0.399959,...,0.211633,0.267488,0.287082,0.198457,0.293935,0.296182,0.275894,0.224054,0.274402,0.222717
Mfap4+ Fibroblasts,0.226879,0.310688,0.350732,0.337923,0.335184,0.252618,0.292011,0.295710,0.358112,0.360570,...,0.205282,0.250665,0.278043,0.219064,0.316651,0.328897,0.257147,0.210824,0.268232,0.239961
Ccl11+ Fibroblasts,0.218387,0.389792,0.483231,0.412661,0.432253,0.311685,0.446480,0.360109,0.436653,0.441058,...,0.210702,0.302529,0.321700,0.219124,0.349897,0.339039,0.276245,0.235772,0.279463,0.252911
Mgp+ Aebp1+ Activated fibroblasts,0.491906,0.359754,0.283063,0.321438,0.327432,0.219296,0.206248,0.269009,0.303973,0.305255,...,0.191115,0.447188,0.482933,0.179963,0.398628,0.257948,0.224912,0.307150,0.408823,0.198636
Il1b+ Fibroblasts,0.272959,0.204179,0.181696,0.199851,0.196478,0.181996,0.170311,0.189662,0.197878,0.195669,...,0.169647,0.226051,0.237110,0.171732,0.212641,0.189108,0.181133,0.192264,0.223703,0.175224


* Save it as a csv file

In [28]:
rss_subcluster.to_csv("rss_subcluster.csv")

# Calculate Regulons Specificity Scores (RSS) Across Fibroblast Subcluster Groups

In [29]:
rss_subcluster_groups = regulon_specificity_scores(auc_mtx=auc_mtx, cell_type_series=meta_data["celltype.group"])

In [30]:
rss_subcluster_groups

,Ahr(+),Alx4(+),Ar(+),Arid3a(+),Arnt(+),Arnt2(+),Arx(+),Atf2(+),Atf3(+),Atf4(+),...,Zfp84(+),Zfp85(+),Zfp874a(+),Zfp93(+),Zfp942(+),Zfp947(+),Zfp952(+),Zfp955a(+),Zscan29(+),Zxdc(+)
Cd248+ Pi16+ Fibroblasts_Obese,0.233717,0.290758,0.287576,0.288882,0.301424,0.268996,0.229516,0.257028,0.296666,0.299587,...,0.203069,0.254690,0.263319,0.195690,0.259617,0.255407,0.234090,0.220067,0.244987,0.209039
Mfap4+ Fibroblasts_Obese,0.213436,0.276768,0.296494,0.294626,0.292471,0.228000,0.255660,0.270642,0.307319,0.307659,...,0.194067,0.234029,0.252773,0.213236,0.282239,0.301105,0.238279,0.203090,0.247129,0.225194
Ccl11+ Fibroblasts_Obese,0.207704,0.330411,0.376296,0.332880,0.348316,0.272133,0.370266,0.308299,0.348226,0.350657,...,0.208654,0.272311,0.283884,0.216548,0.298960,0.301824,0.250928,0.220359,0.252896,0.232431
Mgp+ Aebp1+ Activated fibroblasts_Obese,0.445604,0.316232,0.256714,0.287744,0.293029,0.203904,0.196124,0.247401,0.274371,0.273283,...,0.187521,0.397989,0.423209,0.176378,0.351645,0.239176,0.214080,0.287560,0.366987,0.191089
Il1b+ Fibroblasts_Obese,0.270586,0.203558,0.181227,0.198962,0.195757,0.181809,0.170295,0.189243,0.197110,0.194869,...,0.169676,0.225975,0.236592,0.171788,0.212348,0.188412,0.180757,0.191066,0.223369,0.175306
Mgp+ Aebp1+ Activated fibroblasts_Non-Obese,0.271464,0.245374,0.211858,0.227821,0.228314,0.202382,0.189171,0.212578,0.217843,0.220679,...,0.184590,0.263606,0.277352,0.179356,0.254170,0.211339,0.200201,0.229016,0.253931,0.191069
Mfap4+ Fibroblasts_Non-Obese,0.198529,0.227738,0.254156,0.241433,0.238316,0.216489,0.237619,0.222397,0.251012,0.253329,...,0.198931,0.204235,0.213465,0.197752,0.232372,0.232150,0.215114,0.190807,0.214597,0.210747
Ccl11+ Fibroblasts_Non-Obese,0.194111,0.272314,0.331666,0.295873,0.299970,0.253193,0.320018,0.272371,0.305594,0.307443,...,0.192004,0.236372,0.238309,0.196670,0.261212,0.251336,0.233568,0.212413,0.227575,0.228195
Cd248+ Pi16+ Fibroblasts_Non-Obese,0.198996,0.275988,0.295419,0.293238,0.301315,0.309546,0.204854,0.258535,0.309820,0.310916,...,0.198659,0.209271,0.219726,0.186431,0.231942,0.243743,0.249355,0.195142,0.227660,0.206291
Il1b+ Fibroblasts_Non-Obese,0.173657,0.168869,0.168334,0.169337,0.168946,0.167997,0.167499,0.168560,0.169035,0.169071,...,0.167445,0.168241,0.169194,0.167445,0.168397,0.169065,0.168454,0.171085,0.168758,0.167445


* Save it as a csv file

In [31]:
rss_subcluster_groups.to_csv("rss_subcluster_groups.csv")